In [9]:
import sys
sys.path.insert(0, 'Classes/')
import numpy as np
import pandas as pd
import copy
from Problem  import Problem
from Solution import Solution
from Neighborhood import TransportCostSwap, SourceCostBoost, Reversion
from Algorithm import VariableNeighborhoodSearch
from Execute import RunMultipleMethodsMultipleTimes

## Experimento 1:

Primeiro experimento: Repita esse processo 500 vezes (rodadas)

1. Gera uma solução aleatória

2. Para cada operador de vizinhança, gera 30 vizinhos e fica com o melhor vizinho de cada operador.
Análise:
    - Quantas rodadas cada operador venceu
    - Qual a porcentagem média de melhora de cada


In [10]:
# --- parâmetros de testes ---
num_rounds      = 500    # quantas soluções aleatórias geramos
num_neighbors   = 30     # quantos vizinhos usamos por operador
data = Problem()
data.loadFile("data/data_400.npz")

In [11]:
# os operadores que iremos comparar
operators = [Reversion(2), TransportCostSwap(2, data), SourceCostBoost(2, data),]
op_names = [op.name for op in operators]

In [12]:
# acumular número de vitórias e lista de melhorias
wins         = {name: 0     for name in op_names}
improvements = {name: []    for name in op_names}

for rnd in range(num_rounds):
    # 1) solução inicial aleatória
    sol0 = Solution()
    sol0.generateChromosomeStochastic(data)
    sol0.evaluate(data)
    fx0 = sol0.FX

    # guardo o melhor FX que cada operador conseguiu
    best_fx_by_op = {}

    for op in operators:
        best_fx = fx0
        # 2) gerar N vizinhos e ficar com o melhor
        for _ in range(num_neighbors):
            neigh = op.applyChange(sol0)
            neigh.evaluate(data)
            if neigh.FX < best_fx:
                best_fx = neigh.FX
        best_fx_by_op[op.name] = best_fx

        # 3) calcular porcentagem de melhora (0 se não melhorou)
        imp = max(0.0, (fx0 - best_fx) / fx0 * 100.0)
        improvements[op.name].append(imp)

    # 4) determinar quem “venceu” esta rodada (menor best_fx)
    winner = min(best_fx_by_op, key=best_fx_by_op.get)
    wins[winner] += 1

In [13]:
# --- relatório final ---
print("Número de vitórias por operador:")
for name in op_names:
    print(f"  {name:20s}: {wins[name]:4d} / {num_rounds}")

print("\nMelhoria média (%) por operador:")
for name in op_names:
    avg_imp = np.mean(improvements[name])
    print(f"  {name:20s}: {avg_imp:6.2f}%")

Número de vitórias por operador:
  Reversion           :  449 / 500
  TransportCostSwap   :    1 / 500
  SourceCostBoost     :   50 / 500

Melhoria média (%) por operador:
  Reversion           :   2.36%
  TransportCostSwap   :   0.03%
  SourceCostBoost     :   0.68%


## Experimento 2:

Dada uma solução ótima local obtida por uma execução suficientemente longa do VNS:
1. Gerar 1000 vizinhos através do operador Reversion

2. Gerar 1000 vizinhos através dos dois operadores especializados

3. Comparar:
    - De qual o operador foi o melhor vizinho gerado
    - Quantas vezes cada operador retornou um vizinho melhor
    - Qual a porcentagem média de melhora quando cada operador retorna um vizinho melhor

In [14]:
# --- 1) Obtenha uma solução ótima local com VNS longa ---
data = Problem()
data.loadFile("data/data_400.npz")

vns_long = VariableNeighborhoodSearch(
    [Reversion(2)],
    max_eval=20000,       # avaliações suficientes para buscar ótimo local
    initialization=1,      # 0=determinístico, 1=estocástico
    name="VNS_Long"
)
local_opt = vns_long.solve(data)  # Gera e retorna a melhor solução
fx0 = local_opt.FX
print(f"Custo da solução local ótima: {fx0:.2f}\n")

Initial FX: 388334879.54869604
Final solution: 297103485.69265497
Number of evaluations: 20577
Custo da solução local ótima: 297103485.69



In [15]:
# --- 2) Gerar 1000 vizinhos por operador ---
operators = [
    Reversion(2),
    TransportCostSwap(2, data),
    SourceCostBoost(2, data),
]
names = [op.name for op in operators]

best_neighbor_fx = {name: np.inf for name in names}
count_better      = {name: 0       for name in names}
sum_improvement   = {name: 0.0     for name in names}

num_neighbors = 1000

for op in operators:
    for _ in range(num_neighbors):
        neigh = op.applyChange(local_opt)
        neigh.evaluate(data)
        fxn = neigh.FX
        name = op.name

        # conta quantas vezes melhorou
        if fxn < fx0:
            count_better[name]    += 1
            sum_improvement[name] += (fx0 - fxn) / fx0 * 100

        # registra o melhor vizinho
        if fxn < best_neighbor_fx[name]:
            best_neighbor_fx[name] = fxn

In [16]:
# --- 3) Relatório ---
print("Melhor vizinho (FX) por operador:")
for name in names:
    print(f"  {name:20s}: {best_neighbor_fx[name]:,.2f}")

print("\nNúmero de vizinhos melhores (de 1000):")
for name in names:
    print(f"  {name:20s}: {count_better[name]:4d} / {num_neighbors}")

print("\nMelhora média (%) quando melhorou:")
for name in names:
    cnt = count_better[name]
    if cnt > 0:
        avg_pct = sum_improvement[name] / cnt
        print(f"  {name:20s}: {avg_pct:6.2f}%")
    else:
        print(f"  {name:20s}:    –    (nunca melhorou)")

Melhor vizinho (FX) por operador:
  Reversion           : 296,640,078.01
  TransportCostSwap   : 297,103,485.69
  SourceCostBoost     : 297,014,760.60

Número de vizinhos melhores (de 1000):
  Reversion           :   32 / 1000
  TransportCostSwap   :    0 / 1000
  SourceCostBoost     :   77 / 1000

Melhora média (%) quando melhorou:
  Reversion           :   0.01%
  TransportCostSwap   :    –    (nunca melhorou)
  SourceCostBoost     :   0.02%
